# Notebook 3 ? CVaR-Constrained PPO on DOGEUSDT Replay

In [ ]:
import sys
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()), pathlib.Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

from stable_baselines3 import PPO
from procs.stochastic_processes import MarketReplayMidpriceModel
from procs.gym.calibration import tune_gamma
from procs.gym.cvar_lagrangian import calibrate_cvar_threshold_windowed, train_cvar_ppo
from procs.gym.data_loader import load_single_day
from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.helpers.plotting import plot_cvar_training, plot_trajectory
from procs.gym.notebook_support import (
    build_replay_env,
    evaluate_agent_over_seeds,
    evaluate_as_fast,
    freeze_vecnorm,
    make_vecnorm,
    summarise_agent_frames,
)
from procs.gym.reward_scale import estimate_reward_scale
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.rewards import CjMmCriterion, CjMmDrawdownPenalty, PnLReward
from procs.agents import AvellanedaStoikovAgent, Sb3Agent

## 1. Load Data and Configuration

In [ ]:
cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()

DATA_PATH = cfg.data_path()
S, dt_sec, dt_index = load_single_day(str(DATA_PATH))
T_sec = float(dt_sec.sum())
sigma = MarketReplayMidpriceModel(S, dt_sec).volatility
print(f"Loaded {len(S):,} snapshots from {DATA_PATH.name}, ?={sigma:.6f}, T={T_sec:.0f}s")

## 2. Calibrate A-S Gamma and Reward Scale

In [ ]:
as_gamma, study = tune_gamma(
    midprices=S,
    dt_array=dt_sec,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    gamma_range=cfg.as_gamma_range,
    n_trials=cfg.as_gamma_trials,
    num_trajectories=cfg.as_gamma_num_trajectories,
)
print(f"Using A-S ? = {as_gamma:.6f}")

reward_scale = estimate_reward_scale(
    midprices=S,
    dt_array=dt_sec,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    num_trajectories=cfg.reward_scale_num_trajectories,
    use_bm=False,
)
print(f"Reward scale: {reward_scale:.4f}")

## 3. Shared Replay Environment

In [ ]:
def get_env(reward_fn=None):
    return build_replay_env(S, dt_sec, cfg, reward_fn=reward_fn)

## 4. A-S Baseline

In [ ]:
as_agent = AvellanedaStoikovAgent(as_gamma, sigma, cfg.kappa, T_sec, cfg.tick_size)
seed_range = range(cfg.evaluation_seed, cfg.evaluation_seed + cfg.evaluation_rollouts)
as_eval = evaluate_as_fast(
    S,
    dt_sec,
    gamma=as_gamma,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    q_max=cfg.q_max,
    seeds=seed_range,
)
print(summarise_agent_frames({"A-S Baseline": as_eval}).to_string(float_format="%.6f"))

## 5. Train Unconstrained and DD-Penalised PPO

In [ ]:
%%time
train_uc = get_env(reward_fn=CjMmCriterion(cfg.phi))
sb3_uc = make_vecnorm(StableBaselinesTradingEnvironment(train_uc), cfg, training=True)
model_uc = PPO("MlpPolicy", sb3_uc, verbose=1, device="cpu", **cfg.ppo_kwargs())
model_uc.learn(total_timesteps=cfg.ppo_total_timesteps)
model_uc.save(str(cfg.model_path("ppo_unconstrained_doge")))
sb3_uc.save(str(cfg.vecnorm_path("vecnorm_uc")))
print(f"Saved unconstrained PPO to {cfg.model_path('ppo_unconstrained_doge').with_suffix('.zip')}")
print(f"Saved UC VecNorm to {cfg.vecnorm_path('vecnorm_uc')}")

In [ ]:
%%time
train_dd = get_env(reward_fn=CjMmDrawdownPenalty(cfg.phi, cfg.alpha_dd))
sb3_dd = make_vecnorm(StableBaselinesTradingEnvironment(train_dd), cfg, training=True)
model_dd = PPO("MlpPolicy", sb3_dd, verbose=1, device="cpu", **cfg.ppo_kwargs())
model_dd.learn(total_timesteps=cfg.ppo_total_timesteps)
model_dd.save(str(cfg.model_path("ppo_dd_penalised_doge")))
sb3_dd.save(str(cfg.vecnorm_path("vecnorm_dd")))
print(f"Saved DD-penalised PPO to {cfg.model_path('ppo_dd_penalised_doge').with_suffix('.zip')}")
print(f"Saved DD VecNorm to {cfg.vecnorm_path('vecnorm_dd')}")

## 6. Calibrate the CVaR Threshold from the Unconstrained PPO

In [ ]:
uc_eval_raw = StableBaselinesTradingEnvironment(get_env(PnLReward()))
uc_eval_vn = freeze_vecnorm(cfg.vecnorm_path("vecnorm_uc"), uc_eval_raw, cfg)
ppo_uc_agent = Sb3Agent(model_uc, vecnorm_env=uc_eval_vn)

dd_threshold, cvar_uc = calibrate_cvar_threshold_windowed(
    env=get_env(PnLReward()),
    agent=ppo_uc_agent,
    n_steps=cfg.ppo_n_steps,
    n_windows=50,
    cvar_alpha=0.2,
    tighten=0.20,
)
print(f"CVaR threshold: {dd_threshold:.6f}  (unconstrained CVaR_0.2={cvar_uc:.6f})")

## 7. Train the CVaR-Constrained PPO

In [ ]:
%%time
train_cvar = get_env(reward_fn=CjMmCriterion(cfg.phi))
sb3_cvar = make_vecnorm(StableBaselinesTradingEnvironment(train_cvar), cfg, training=True)
model_cvar, cb_cvar, _ = train_cvar_ppo(
    sb3_env=sb3_cvar,
    total_timesteps=cfg.ppo_total_timesteps,
    cvar_alpha=0.2,
    dd_threshold=dd_threshold,
    lr_lambda=0.01,
    lambda_max=500.0,
    model_init=model_uc,
    ppo_kwargs=cfg.ppo_kwargs(),
    verbose=0,
)
model_cvar.save(str(cfg.model_path("ppo_cvar_doge")))
sb3_cvar.save(str(cfg.vecnorm_path("vecnorm_cvar")))
print(f"CVaR PPO trained. Final ?={cb_cvar.lambda_:.4f}")

In [ ]:
plot_cvar_training(cb_cvar.cvar_history, cb_cvar.lambda_history, dd_threshold)

## 8. Compare All Four Agents on the Same Evaluation Protocol

In [ ]:
agents = {
    "Unconstrained PPO": Sb3Agent(
        model_uc,
        vecnorm_env=freeze_vecnorm(
            cfg.vecnorm_path("vecnorm_uc"),
            StableBaselinesTradingEnvironment(get_env(PnLReward())),
            cfg,
        ),
    ),
    "DD-Penalised PPO": Sb3Agent(
        model_dd,
        vecnorm_env=freeze_vecnorm(
            cfg.vecnorm_path("vecnorm_dd"),
            StableBaselinesTradingEnvironment(get_env(PnLReward())),
            cfg,
        ),
    ),
    "CVaR PPO": Sb3Agent(
        model_cvar,
        vecnorm_env=freeze_vecnorm(
            cfg.vecnorm_path("vecnorm_cvar"),
            StableBaselinesTradingEnvironment(get_env(PnLReward())),
            cfg,
        ),
    ),
}

agent_frames = {"A-S Baseline": as_eval}
for name, agent in agents.items():
    agent_frames[name] = evaluate_agent_over_seeds(
        lambda: get_env(PnLReward()),
        agent,
        seeds=seed_range,
    )

comparison = summarise_agent_frames(agent_frames)
comparison.to_csv(cfg.result_path("nb3_four_agent_summary.csv"))
print(comparison.to_string(float_format="%.6f"))
print(f"Saved summary to {cfg.result_path('nb3_four_agent_summary.csv')}")

## 9. Representative Trajectories

In [ ]:
trajectory_agents = {"A-S Baseline": as_agent, **agents}
for name, agent in trajectory_agents.items():
    print(f"\n{'=' * 60}\n  {name}\n{'=' * 60}")
    plot_trajectory(get_env(PnLReward()), agent, seed=cfg.evaluation_seed, datetime_index=dt_index)
